# 실험 제목
- 담당: 김영빈
- 날짜: 26/09/25
- 목적: RAG 문서를 임베딩해서 pgvector에 저장하고, 질문으로 top-k 검색까지 확인 (1단계: 임베딩 모델 동작 확인)

> 끝나면 결과를 `experiments/LOG.md`에 한 줄 남기기

## 1단계. 임베딩 모델 로드

`dragonkue/snowflake-arctic-embed-l-v2.0-ko`를 sentence-transformers로 불러온다.

이 모델은 **질문(query)과 문서(document)를 다르게 인코딩해야 하는 비대칭(asymmetric) 모델**이다:
- 질문: `model.encode(text, prompt_name="query")` — 검색에 최적화된 특수 프롬프트가 내부적으로 붙음
- 문서: `model.encode(text)` — 접두사 없이 그대로

이 차이를 지키지 않으면 같은 모델을 쓰고도 검색 품질이 떨어진다(모델 카드에 명시된 내용).
처음 실행하면 허깅페이스에서 모델 가중치를 다운로드하므로 몇 분 걸릴 수 있다.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "dragonkue/snowflake-arctic-embed-l-v2.0-ko"

print("모델 로드 중...")
model = SentenceTransformer(EMBEDDING_MODEL)
print("완료. 임베딩 차원:", model.get_embedding_dimension())

## 2단계. 질문 vs 문서 인코딩이 실제로 다른지, 결과가 그럴듯한지 확인

같은 주제(대출 만기 연장)의 질문 1개와 관련 있는 문서 1개, 그리고 관련 없는 문서(보험금 청구) 1개를 인코딩해서
코사인 유사도를 비교한다. **관련 있는 문서 쪽 유사도가 더 높게 나와야** 모델이 제대로 동작하는 것이다.

`normalize_embeddings=True`로 벡터를 정규화하면 내적(dot product)이 곧 코사인 유사도가 된다
(모델 카드에서 권장하는 방식, pgvector의 `vector_cosine_ops`와도 맞는 방식).

In [ ]:
import numpy as np

sample_query = "대출 만기를 연장하려면 어떻게 해야 하나요?"
sample_doc_relevant = (
    "[요구사항] 대출 만기 연장 절차를 안내하시오.\n"
    "[고객 질문] 대출 만기가 다가오는데 연장하려면 어떻게 해야 하나요?"
)
sample_doc_irrelevant = (
    "[요구사항] 보험금 청구 절차를 안내하시오.\n"
    "[고객 질문] 자동차 사고가 났는데 보험금 청구는 어떻게 하나요?"
)

# 질문은 prompt_name="query" 를 붙여서 인코딩 (검색용 특수 프롬프트)
query_emb = model.encode(sample_query, prompt_name="query", normalize_embeddings=True)
# 문서는 접두사 없이 그대로 인코딩
doc_relevant_emb = model.encode(sample_doc_relevant, normalize_embeddings=True)
doc_irrelevant_emb = model.encode(sample_doc_irrelevant, normalize_embeddings=True)

def cosine(a, b):
    return float(np.dot(a, b))  # 정규화된 벡터라 내적 = 코사인 유사도

print(f"질문 벡터 차원: {query_emb.shape}")
print(f"[관련 있음] 대출 연장 문서와 유사도: {cosine(query_emb, doc_relevant_emb):.4f}")
print(f"[관련 없음] 보험금 청구 문서와 유사도: {cosine(query_emb, doc_irrelevant_emb):.4f}")

## 3단계. pgvector 연결 + `document_chunk` 테이블 생성

DB에 아래 구조의 테이블을 만든다. `text`(임베딩 대상 원문)와 `embedding`(벡터)을 같은 행에 둬서,
검색 결과에서 바로 원문을 같이 받을 수 있게 한다. 메타데이터는 회의에서 합의한 4개만 컬럼으로 둔다.

```
document_chunk
├─ doc_id              TEXT PRIMARY KEY   원본 QA ID. 중복 방지 + 재실행 시 upsert 기준
├─ text                TEXT               요구사항+질문+답변+꼬리질문+종합답변을 합친 임베딩 대상 원문
├─ category             TEXT               은행 / 보험 / 증권
├─ consulting_topic     TEXT               세부 상담 주제
├─ qa_topic             TEXT               QA 단위 주제
├─ consulting_purpose   TEXT               상담 목적
└─ embedding            VECTOR(1024)       임베딩 벡터 (1단계에서 확인한 모델 출력 차원)
```

인덱스(HNSW)는 아직 안 만든다 — 데이터를 다 넣은 뒤에 만들어야 빠르다
(먼저 만들면 문서를 넣을 때마다 검색 그래프를 다시 계산해서 느려진다). 이건 5단계에서 한다.

In [ ]:
import os

import psycopg
from dotenv import load_dotenv
from pgvector.psycopg import register_vector

load_dotenv()  # .env 에서 DATABASE_URL 읽기 (docker-compose 로 띄운 로컬 pgvector)

EMBEDDING_DIM = model.get_embedding_dimension()  # 1단계에서 확인한 1024

conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True)

with conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector")  # pgvector 확장 먼저 활성화 (이미 있으면 아무 일도 안 함)

register_vector(conn)  # 확장이 있어야 psycopg 가 vector 타입을 인식할 수 있음 (그래서 extension 생성 다음에 호출)

with conn.cursor() as cur:
    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS document_chunk (
            doc_id TEXT PRIMARY KEY,
            text TEXT NOT NULL,
            category TEXT,
            consulting_topic TEXT,
            qa_topic TEXT,
            consulting_purpose TEXT,
            embedding VECTOR({EMBEDDING_DIM})
        )
    """)

print("테이블 생성 완료 (또는 이미 존재)")

## 4단계. 테이블 생성 확인

DB에 실제로 어떤 컬럼이 만들어졌는지 조회해서 눈으로 확인한다 (DBeaver를 새로고침해서 봐도 동일하게 보인다).

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = \'document_chunk\'
        ORDER BY ordinal_position
    """)
    for column_name, data_type in cur.fetchall():
        print(f"  {column_name:<20} {data_type}")

## 관찰 / 메모
- 임베딩 차원 1024 확인 (설계대로)
- 관련 있는 문서(대출 연장) 유사도 0.7370 vs 관련 없는 문서(보험금 청구) 유사도 0.1760 — 격차가 커서 질문/문서 비대칭 인코딩이 제대로 동작하는 것으로 판단
- 모델은 로컬 캐시에 저장돼서 다음 실행부터는 다운로드 없이 빠르게 로드됨
